# Cairo Weather — Time-Series Preprocessing

This notebook converts the Task 4 EDA dataset into a leakage-safe, model-ready daily time series. The source was intentionally sampled every two days, so its native modeling frequency is **2 days (`2D`)**—not hourly. Calendar, lag, and rolling features use period counts at this frequency.

**Target:** `target_temperature_c` (daily mean 2 m air temperature, °C)  
**Forecast split:** final 21 calendar days, kept strictly after the training period.

In [3]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Preprocessing' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'EDA' / 'dataset' / 'cairo_weather_2021_2026_2day.csv'
OUTPUT_DIR = PROJECT_ROOT / 'Preprocessing'
OUTPUT_DIR.mkdir(exist_ok=True)
TARGET = 'target_temperature_c'
DATA_PATH

WindowsPath('D:/ML_Intern/Cairo Weather Forecast/Preprocessing/notebook/EDA/dataset/cairo_weather_2021_2026_2day.csv')

## 1. Load, validate, and align the time index

We first remove duplicate timestamps, sort chronologically, and explicitly restore the expected `2D` grid. This exposes any missing observation dates without converting the deliberately two-day source into fabricated daily observations.

In [4]:
# Resolve the dataset path robustly in case the notebook is launched from a different folder
candidate_paths = [
        DATA_PATH,
        PROJECT_ROOT / 'EDA' / 'dataset' / 'cairo_weather_2021_2026_2day.csv',
        PROJECT_ROOT.parent / 'EDA' / 'dataset' / 'cairo_weather_2021_2026_2day.csv',
        PROJECT_ROOT.parent.parent / 'EDA' / 'dataset' / 'cairo_weather_2021_2026_2day.csv',
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), DATA_PATH)

if not DATA_PATH.exists():
        raise FileNotFoundError(
                f"Could not find dataset. Checked:\n" + "\n".join(str(p) for p in candidate_paths)
        )

raw = pd.read_csv(DATA_PATH, parse_dates=['date'])
assert TARGET in raw.columns, f'Missing required target: {TARGET}'

df = (raw.drop_duplicates(subset='date', keep='last')
        .set_index('date')
        .sort_index())
df.index.name = 'date'

observed_deltas = df.index.to_series().diff().dropna()
native_freq = observed_deltas.mode().iloc[0]
assert native_freq == pd.Timedelta(days=2), f'Unexpected dominant sampling interval: {native_freq}'
FREQ = '2D'
expected_index = pd.date_range(df.index.min(), df.index.max(), freq=FREQ)
missing_timestamps = expected_index.difference(df.index)
df = df.reindex(expected_index)
df.index.name = 'date'

print(f'Rows before alignment: {len(raw):,}')
print(f'Native frequency: {FREQ} ({native_freq.days} calendar days)')
print(f'Missing timestamps introduced/exposed: {len(missing_timestamps):,}')
print(f'Index frequency: {df.index.freqstr}')
df[[TARGET]].head()

Rows before alignment: 1,055
Native frequency: 2D (2 calendar days)
Missing timestamps introduced/exposed: 0
Index frequency: 2D


,target_temperature_c
date,
2021-01-01,14.84
2021-01-03,15.12
2021-01-05,17.09
2021-01-07,18.22
2021-01-09,15.94


## 2. Missing values and time-aware imputation

Short interior gaps are interpolated by time for smooth numeric weather variables (at most two `2D` periods / four days). Longer target gaps use an annual seasonal-naive value from roughly the same date in the prior year (`183` two-day periods); anything still unavailable is left missing and reported. Radiation columns are not used as target predictors here, so their trailing missing values remain visible rather than being invented.

In [5]:
missing_before = df.isna().sum()
numeric_cols = df.select_dtypes(include='number').columns.tolist()
derived_from_eda = ['year', 'month', 'day_of_year', 'day_of_week', 'is_weekend',
                    'temperature_range_c', 'dewpoint_depression_c', 'wind_u_10m_m_s',
                    'wind_v_10m_m_s', 'day_of_year_sin', 'day_of_year_cos']
weather_cols = [c for c in numeric_cols if c not in derived_from_eda]

# Interpolate only short *internal* gaps. limit_area prevents edge extrapolation.
df[weather_cols] = df[weather_cols].interpolate(
    method='time', limit=2, limit_area='inside'
)

SEASONAL_PERIODS = 183  # ~366 days / 2-day native frequency
long_target_gaps = df[TARGET].isna()
df.loc[long_target_gaps, TARGET] = df[TARGET].shift(SEASONAL_PERIODS)[long_target_gaps]

missing_report = pd.DataFrame({
    'before': missing_before,
    'after_short_gap_policy': df.isna().sum()
}).query('before > 0 or after_short_gap_policy > 0').sort_values('before', ascending=False)
display(missing_report)
assert df[TARGET].notna().all(), 'Target still has gaps; inspect source dates before modeling.'

,before,after_short_gap_policy
all_sky_shortwave_mj_m2_day,36,36
all_sky_longwave_mj_m2_day,36,36
clear_sky_shortwave_mj_m2_day,36,36


## 3. Physical validation and rolling outlier caps

Impossible humidity, precipitation, and wind values are replaced with missing values before the same short-gap policy is used. Target-temperature outliers are identified with a centered 31-period rolling median/MAD rule, then winsorized (capped) instead of removed so the series remains continuous. The original target is preserved for auditability.

In [6]:
df['target_temperature_original_c'] = df[TARGET]
physical_rules = {
    'relative_humidity_pct': (0, 100),
    'precipitation_mm_day': (0, np.inf),
    'wind_speed_10m_m_s': (0, np.inf),
    'wind_speed_50m_m_s': (0, np.inf),
}
invalid_counts = {}
for col, (low, high) in physical_rules.items():
    if col in df:
        invalid = ~df[col].between(low, high) & df[col].notna()
        invalid_counts[col] = int(invalid.sum())
        df.loc[invalid, col] = np.nan

window = 31
rolling_median = df[TARGET].rolling(window, center=True, min_periods=15).median()
rolling_mad = (df[TARGET] - rolling_median).abs().rolling(window, center=True, min_periods=15).median()
robust_sigma = 1.4826 * rolling_mad
lower, upper = rolling_median - 4 * robust_sigma, rolling_median + 4 * robust_sigma
outlier_mask = ((df[TARGET] < lower) | (df[TARGET] > upper)).fillna(False)
df[TARGET] = df[TARGET].clip(lower=lower, upper=upper)

# Repair only short gaps that physical validation may have created.
repair_cols = [c for c in physical_rules if c in df]
df[repair_cols] = df[repair_cols].interpolate(method='time', limit=2, limit_area='inside')
print('Invalid physical values:', invalid_counts)
print(f'Target observations capped by rolling MAD rule: {outlier_mask.sum()}')

Invalid physical values: {'relative_humidity_pct': 0, 'precipitation_mm_day': 0, 'wind_speed_10m_m_s': 0, 'wind_speed_50m_m_s': 0}
Target observations capped by rolling MAD rule: 6


## 4. Forecast-safe features

Calendar encodings are known at forecast time. Lags and rolling values use only current/past observations (`shift(1)` before rolling) to avoid using the value being predicted. With `2D` data, `lag_1` is two days ago, `lag_12` is about 24 days ago, and `lag_183` is about one year ago.

In [7]:
df['month'] = df.index.month
df['day_of_week'] = df.index.dayofweek
df['is_weekend'] = (df.index.dayofweek >= 5).astype('int8')
df['day_of_year'] = df.index.dayofyear
df['day_of_year_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
df['day_of_year_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)

for lag in (1, 12, SEASONAL_PERIODS):
    df[f'{TARGET}_lag_{lag}'] = df[TARGET].shift(lag)
for periods in (4, 15, 31):
    history = df[TARGET].shift(1)
    df[f'{TARGET}_roll_mean_{periods}'] = history.rolling(periods, min_periods=periods).mean()
    df[f'{TARGET}_roll_std_{periods}'] = history.rolling(periods, min_periods=periods).std()

feature_cols = [
    'day_of_year_sin', 'day_of_year_cos', 'day_of_week', 'is_weekend',
    f'{TARGET}_lag_1', f'{TARGET}_lag_12', f'{TARGET}_lag_{SEASONAL_PERIODS}',
    *[f'{TARGET}_roll_{stat}_{p}' for p in (4, 15, 31) for stat in ('mean', 'std')]
]
model_df = df.dropna(subset=[TARGET, *feature_cols]).copy()
print(f'Model-ready rows after feature warm-up: {len(model_df):,} / {len(df):,}')
model_df[[TARGET, *feature_cols[:5]]].head()

Model-ready rows after feature warm-up: 872 / 1,055


,target_temperature_c,day_of_year_sin,day_of_year_cos,day_of_week,is_weekend,target_temperature_c_lag_1
date,,,,,,
2022-01-02,12.24,0.034398,0.999408,6,1,12.72
2022-01-04,11.98,0.068755,0.997634,1,0,12.24
2022-01-06,13.61,0.103031,0.994678,3,0,11.98
2022-01-08,13.50,0.137185,0.990545,5,1,13.61
2022-01-10,10.57,0.171177,0.985240,0,0,13.50


## 5. Chronological split and LSTM scaling

The last 21 calendar days are the hold-out set. No rows are shuffled. SARIMA should use `train[TARGET]` in original °C units; only neural-network inputs need scaling. The scaler below is fit on training data alone.

In [8]:
TEST_DAYS = 21
split_date = model_df.index.max() - pd.Timedelta(days=TEST_DAYS)
train = model_df.loc[model_df.index <= split_date].copy()
test = model_df.loc[model_df.index > split_date].copy()
assert train.index.max() < test.index.min()
assert len(test) > 0

scaler = MinMaxScaler()
train_target_scaled = scaler.fit_transform(train[[TARGET]])
test_target_scaled = scaler.transform(test[[TARGET]])

print(f'Train: {train.index.min().date()} to {train.index.max().date()} ({len(train)} rows)')
print(f'Test:  {test.index.min().date()} to {test.index.max().date()} ({len(test)} rows)')
print('Scaler fit range (training target only):', scaler.data_min_[0], 'to', scaler.data_max_[0])

Train: 2022-01-02 to 2026-09-18 (861 rows)
Test:  2026-09-20 to 2026-10-10 (11 rows)
Scaler fit range (training target only): 6.57 to 34.92


## 6. Export artifacts and preprocessing audit

The exported CSV retains all original columns plus preprocessing fields and features. `model_ready` is a compact modeling table. The JSON audit records the frequency, imputation policy, outlier treatment, and split boundary for reproducibility.

In [9]:
processed_path = OUTPUT_DIR / 'cairo_weather_preprocessed_2day.csv'
model_ready_path = OUTPUT_DIR / 'cairo_weather_model_ready_2day.csv'
audit_path = OUTPUT_DIR / 'preprocessing_audit.json'

df.to_csv(processed_path, index_label='date')
model_df[[TARGET, *feature_cols]].to_csv(model_ready_path, index_label='date')
audit = {
    'source_file': str(DATA_PATH),
    'target': TARGET,
    'frequency': FREQ,
    'seasonal_periods': SEASONAL_PERIODS,
    'missing_timestamps_exposed': int(len(missing_timestamps)),
    'short_gap_interpolation_limit_periods': 2,
    'target_outliers_capped': int(outlier_mask.sum()),
    'physical_invalid_counts': invalid_counts,
    'train_end': str(train.index.max().date()),
    'test_start': str(test.index.min().date()),
    'test_end': str(test.index.max().date()),
}
audit_path.write_text(json.dumps(audit, indent=2), encoding='utf-8')

display(pd.Series(audit))
print(f'\nSaved: {processed_path.name}, {model_ready_path.name}, {audit_path.name}')

source_file                              D:\ML_Intern\Cairo Weather Forecast\EDA\datase...
target                                                                target_temperature_c
frequency                                                                               2D
seasonal_periods                                                                       183
missing_timestamps_exposed                                                               0
short_gap_interpolation_limit_periods                                                    2
target_outliers_capped                                                                   6
physical_invalid_counts                  {'relative_humidity_pct': 0, 'precipitation_mm...
train_end                                                                       2026-09-18
test_start                                                                      2026-09-20
test_end                                                                        2026-10-10


Saved: cairo_weather_preprocessed_2day.csv, cairo_weather_model_ready_2day.csv, preprocessing_audit.json
